In [ ]:
import os
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from plotActivity import plotActivity

In [ ]:
path_to_outdir = './data/mSigAct/'

### Calculate prop from prop1 for custom prop in mSigAct


In [ ]:
data_num_mutations = pd.DataFrame()
for exp in glob.glob(path_to_outdir + 'output/raw_output/all_relatable_sbs_prop1/*/*.exposure.csv'):
    df_exp = pd.read_csv(exp)
    df_exp['Type'] = df_exp.columns[1]
    df_exp = df_exp.rename(columns={df_exp.columns[0]:"SBS", df_exp.columns[1]:'NumMut'})
    data_num_mutations = pd.concat([data_num_mutations, df_exp], axis=0)

data_num_mutations.head()

In [ ]:
sum_sbs = data_num_mutations.groupby('SBS').agg({'NumMut': 'sum'}).reset_index()
sum_sbs['prop'] = sum_sbs['NumMut'] / sum_sbs['NumMut'].sum()
sum_sbs = sum_sbs.sort_values(by='prop', ascending=False)
sum_sbs.head(6) ### They are used in our 3d test with custom prop from 1

### After running mSigAct with custom prop from 1 we can continue running this notebook

Make table for all test from mSigAct

In [ ]:
for test_prop in ['custom_prop', 'all_relatable_sbs_prop1', 'custom_from1']:
    
    data_collect = pd.DataFrame() # tmp

    for exp in glob.glob(path_to_outdir + f'output/raw_output/{test_prop}/*/*.exposure.csv'):

        df_exp = pd.read_csv(exp)
        df_exp['Samples'] = df_exp.columns[1]
        df_exp = df_exp.rename(columns={df_exp.columns[0]:"SBS", df_exp.columns[1]:'NumMut'})
        df_exp = df_exp.pivot(index='Samples', columns='SBS', values='NumMut').reset_index()
        data_collect = pd.concat([data_collect, df_exp], axis=0)
        data_collect = data_collect.fillna(0)
    
    data_collect.to_csv(path_to_outdir + f'output/{test_prop}_Activities.txt', index=False, sep='\t')
    

collect distances of recontructed spectra after assignment

In [ ]:
# collect distances of recontructed spectra after assignment
data_distances  = []
for distances_file in glob.glob(path_to_outdir + 'output/raw_output/*/*/*.distances.csv'):
    dist_df = pd.read_csv(distances_file, index_col=0).assign(
        params=distances_file.split('/')[-3],
        sample=distances_file.split('/')[-2],
    )
    data_distances.append(dist_df)
distances = pd.concat(data_distances)
distances.set_index(['params', 'sample']).to_csv('./data/mSigAct/output/distances.csv')
distances
distances.to_csv(path_to_outdir + 'output/Distances_mSigAct.csv')

In [ ]:
distances[distances.method == 'cosine']

In [ ]:
# prop1 == prop05 (prop05 covered by prop1)
sns.kdeplot(data=distances[(distances.method == 'cosine')].reset_index(), x='QP.assignment', hue='params');
sns.histplot(data=distances[(distances.method == 'cosine')].reset_index(), x='QP.assignment', hue='params');

In [ ]:
#collect activities after assignment
data_activities = []

try_d_full_path = os.path.join(path_to_outdir, 'output/raw_output')
if os.path.isdir(try_d_full_path):
    d = []
    for activity_file in glob.glob(try_d_full_path + '/*/*/*.exposure.csv'):
        act_df = pd.read_csv(activity_file, index_col=0)
        d.append(act_df)
    try_act_df = pd.concat(d, axis=1).assign(params=try_d)
    data_activities.append(try_act_df)

activities = pd.concat(data_activities, axis=0)
activities.index.name = 'signature'
activities = activities.reset_index().set_index(['params', 'signature'])
activities.to_csv('./data/mSigAct/output/activities_total.csv')
activities.head()

In [ ]:
custom_colors = [
        '#acf2d0', 
        '#63d69e', 
        '#f8b6b3', 
        '#d9f7b0', 
        '#fae1a5', 
        '#fad682', 
        'tab:pink', 
        'tab:orange', 
        'tab:olive', 
        'tab:purple', 
        '#7852d9',
    ]

### Plot mSigAct with custom function
TODO: custom colors

In [ ]:
msig_act_list = glob.glob('./data/mSigAct/output/*Activities.txt')
for msig_file in msig_act_list:
    activity_name = msig_file.split('/')[-1].split('Activities')[0]
    plotActivity(msig_file, output_file=path_to_outdir + f'output/figures/{activity_name}mSigAct.pdf')

In [ ]:
d = activities.melt(ignore_index=False, var_name='sample', value_name='activity').reset_index()
d

In [ ]:
sns.boxplot(data=d, x='signature', y='activity')

In [ ]:
sns.boxplot(data=d, x='sample', y='activity', hue='signature')